# Institution Website RAG Chatbot

**RAG-based chatbot from a school or institution website — crawl, index, and chat with source citations.**

---

## What it does

This exercise takes a **specified school or institution website** (e.g. `BASE_URL` in config), **crawls** it (same-domain, link limit), builds a **RAG knowledge base** (Chroma + embeddings, with optional BERTopic categorization and LLM or splitter chunking), and provides an **interactive chatbot** so visitors can learn from the institution; every answer cites source pages.

---

## What I Learned in Week 5

Week 5 covered the core building blocks of RAG (Retrieval-Augmented Generation):
- **RAG fundamentals**: Combining LLM with knowledge base for accurate Q&A
- **Chroma vector database**: Storing embeddings for semantic search
- **Text embeddings**: Converting text to numerical representations
- **Document chunking**: Splitting documents for optimal retrieval
- **Semantic search**: Finding relevant docs by meaning, not keywords
- **Evaluation**: Measuring RAG system accuracy (evals)
- **LangChain 1.0**: Modern RAG pipeline construction

## Why This Project

A tool that can:
1. Answer questions about the school or institution from its website
2. Cite website sources for every answer
3. Work with the institution’s own content (no API training)
4. Provide accurate answers grounded in crawled content
5. Run locally for privacy (optional)

---

## Components Used

| Component | Purpose |
|---|---|
| `crawler` (requests + BeautifulSoup) | Same-domain crawl of institution site, 200-link limit, stats |
| `bertopic` + `transformers` (zero-shot) | BERTopic topics, zero-shot category names (configurable models) |
| `chromadb` | Persistent vector database at data/db |
| `OpenAIEmbeddings` | Convert text to vectors |
| `RecursiveCharacterTextSplitter` or LLM chunking | Document chunking (set `USE_LLM_CHUNKING`) |
| Configurable LLM clients | Chunking, reranking, and chat (model / base_url / api_key per task; OpenAI or Ollama) |
| `evaluation/` (tests.jsonl, eval) | Retrieval + answer evals (keyword coverage, LLM judge) |
| `gr.Blocks` + Tabs | Chat UI with streaming, Sources tab (extracted docs + links), Evaluation tab with charts |

---

## How It Works

1. **Crawl** → Visit institution/school website, follow same-domain links (max 200), extract content; BERTopic on all pages for content-based categories (zero-shot topic names). Re-run to re-categorize existing crawl.
2. **Store** → Save to `data/source/<domain>/<category>/` with source URL per page; optional _summary.md
3. **Load & Chunk** → Read from data/source, split into semantic sections (~500 chars each)
4. **Embed** → Convert chunks to vectors using OpenAI
5. **Store** → Save to Chroma at `data/db`
6. **Query** → User asks question, convert to embedding
7. **Retrieve** → Find similar chunks using cosine similarity
8. **Generate** → Pass context + question to LLM
9. **Respond** → Answer with source URL citations

---

## RAG Pipeline Architecture

```
[User Query]
       ↓
[Embed Query] → OpenAI Embeddings
       ↓
[Semantic Search] → Chroma (RETRIEVAL_K chunks)
       ↓
[LLM Re-rank] → Top RERANK_TOP_K chunks
       ↓
[Generate Answer] → LLM with context
       ↓
[Response + Sources] → Final output
```

Context is the crawled institution site content. Set `USE_LLM_CHUNKING = True` for LLM-based chunking. Use the **Evaluation** tab in the Gradio UI (or Cell 6b) to run retrieval (MRR, keyword coverage) and answer (LLM-as-judge) metrics over `evaluation/tests.jsonl`. Chat uses streaming; Sources tab shows extracted docs with links for the last response.

---

**Hardware:** Runs on CPU with optional GPU for embeddings

## Cell 1 — Install Dependencies

In [ ]:
# Run once — restart kernel after installation
!uv pip install -q chromadb langchain-openai langchain-core tiktoken gradio plotly scikit-learn requests beautifulsoup4 transformers bertopic sentence-transformers umap-learn hdbscan tqdm

print("Dependencies installed. Restart kernel if needed.")

## Cell 2 — Configuration & Imports

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
import gradio as gr




# Configuration
load_dotenv(override=True)

DEFAULT_MODEL = "gpt-5-nano"
OLLAMA_MODEL = "llama3.2"

openai_api_key = os.getenv("OPENAI_API_KEY", "")
if openai_api_key:
    print(f"OpenAI API Key: {openai_api_key[:8]}...")

else:
    DEFAULT_MODEL = OLLAMA_MODEL
    print("⚠️ No OpenAI key found!")

CHAT_MODEL = "gpt-5-nano"
CHUNKING_MODEL = "gpt-5-nano"
RERANKING_MODEL = "gpt-5-nano"


OLLAMA_BASE_URL = "http://localhost:11434/v1"



def get_openai_client(client_type: str):
    if not openai_api_key:
        return OpenAI(base_url=OLLAMA_BASE_URL, model=OLLAMA_MODEL,api_key="ollama"), OLLAMA_MODEL
    if client_type == "chat":
        return OpenAI(api_key=openai_api_key), CHAT_MODEL
    elif client_type == "chunking":
        return OpenAI(api_key=openai_api_key), CHUNKING_MODEL
    elif client_type == "reranking":
        return OpenAI(api_key=openai_api_key), RERANKING_MODEL



EMBEDDING_MODEL = "text-embedding-3-small"
COLLECTION_NAME = "documents"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
RETRIEVAL_K = 10
RERANK_TOP_K = 3
USE_LLM_CHUNKING = True


SENTENCE_TRANSFORMER_MODEL = "all-MiniLM-L6-v2"
ZERO_SHOT_CLASSIFIER_MODEL = "typeform/distilbert-base-uncased-mnli"
HDBSCAN_MIN_CLUSTER_SIZE = 4
HDBSCAN_MIN_SAMPLES = 2

DATA_DIR = Path("data")
DATA_SOURCE = DATA_DIR / "source"
DB_PATH = DATA_DIR / "db"
BASE_URL = "https://www.uniport.edu.ng"




embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

from evaluation.eval import set_judge_client

_judge_client, _judge_model = get_openai_client("chat")
set_judge_client(_judge_client, _judge_model)


print("✅ Configuration ready.", {
    "default_model": DEFAULT_MODEL,
    "ollama_model": OLLAMA_MODEL,
    "openai_model": CHAT_MODEL,
    "embedding_model": EMBEDDING_MODEL,
    "collection_name": COLLECTION_NAME,
    
})

## Cell 3 — Knowledge Base from Website Crawl

### Crawl and content-based categorization

Same-domain crawl of the institution site (link limit), then BERTopic + zero-shot classifier to assign content-based categories. Re-run to re-categorize existing pages.

In [ ]:
from urllib.parse import urlparse
import re
import shutil
import numpy as np
from crawler import crawl_site
from tqdm.auto import tqdm
from transformers import pipeline
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from hdbscan import HDBSCAN

def get_domain(url: str) -> str:
    return urlparse(url).netloc or "unknown"

def sanitize_category(name: str) -> str:
    s = (name or "").lower().replace(" ", "_").replace("-", "_").strip()
    s = re.sub(r"[^a-z0-9_]", "", s)[:30]
    return s or "general"


def slug_from_title(title: str, url: str = "", max_len: int = 35) -> str:
    raw = (title or "").strip()
    if not raw or raw.lower() == "no title":
        if url:
            path = urlparse(url).path.strip("/")
            raw = path.split("/")[-1] if path else "page"
        else:
            raw = "page"
    s = raw.lower().replace(" ", "_").replace("-", "_")
    s = re.sub(r"[^a-z0-9_]", "", s).strip("_")[:max_len]
    return s or "page"

CATEGORY_STOPLIST = frozenset({
    "and", "the", "for", "are", "but", "not", "you", "all", "can", "had", "her", "his", "was", "one",
    "our", "out", "has", "how", "its", "may", "now", "own", "say", "see", "way", "who", "did", "get",
    "click", "read", "more", "here", "page", "form", "back", "next", "prev", "view", "link", "menu",
})
MIN_CATEGORY_LEN = 3

def is_number_or_contact_like(sanitized: str) -> bool:
    if not sanitized:
        return False
    return all(c.isdigit() for c in sanitized)

def is_meaningful_category(sanitized: str) -> bool:
    if len(sanitized) < MIN_CATEGORY_LEN or sanitized in CATEGORY_STOPLIST:
        return False
    if is_number_or_contact_like(sanitized):
        return False
    return True

def filter_topic_words(words_tuples: list, max_words: int = 5) -> list:
    if not words_tuples:
        return []
    allowed = [w for w, _ in words_tuples if len(w) >= MIN_CATEGORY_LEN and w.lower() not in CATEGORY_STOPLIST and not is_number_or_contact_like(re.sub(r"[^a-z0-9]", "", w.lower()))]
    return allowed[:max_words]

def load_pages_from_md(domain_dir: Path) -> list[dict]:
    pages = []
    for md_path in domain_dir.rglob("*.md"):
        if md_path.name == "_summary.md":
            continue
        raw = md_path.read_text(encoding="utf-8")
        url, title, text = "", "No title", raw
        if raw.startswith("Source: "):
            first_line, _, rest = raw.partition("\n\n")
            url = first_line.replace("Source: ", "").strip()
            parts = rest.split("\n\n", 1)
            if parts and parts[0].startswith("# "):
                title = parts[0].replace("# ", "").strip()
                text = parts[1] if len(parts) > 1 else ""
            else:
                text = rest
        pages.append({"url": url, "title": title, "text": text})
    return pages

def categorize_and_save(pages: list[dict], domain_dir: Path) -> None:
    docs_for_bertopic = [(p["title"] + " " + (p["text"] or "")[:500])[:512] for p in pages]
    hdbscan_model = HDBSCAN(
        min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
        min_samples=HDBSCAN_MIN_SAMPLES,
        cluster_selection_method="eom",
        prediction_data=True,
    )
    embedding_model = SentenceTransformer(SENTENCE_TRANSFORMER_MODEL)
    topic_model = BERTopic(embedding_model=embedding_model, hdbscan_model=hdbscan_model)
    topics, probs = topic_model.fit_transform(docs_for_bertopic)
    topics = np.asarray(topics)
    if probs is not None:
        probs = np.asarray(probs)
    classifier = pipeline("zero-shot-classification", model=ZERO_SHOT_CLASSIFIER_MODEL)
    topic_label_map = {-1: "general"}
    unique_topics = sorted(set(topics))
    for topic_id in unique_topics:
        if topic_id == -1:
            continue
        words_tuples = topic_model.get_topic(topic_id) or []
        words = filter_topic_words(words_tuples, max_words=5)
        if not words:
            topic_label_map[topic_id] = "general"
            continue
        indices = np.where(topics == topic_id)[0]
        if probs is not None and probs.ndim == 2 and topic_id < probs.shape[1]:
            j = int(indices[np.argmax(probs[indices, topic_id])])
        else:
            j = int(indices[0])
        rep_doc = docs_for_bertopic[j]
        out = classifier(rep_doc, words, multi_label=False)
        chosen = "general"
        for label in out["labels"]:
            cand = sanitize_category(label)
            if is_meaningful_category(cand):
                chosen = cand
                break
        if is_number_or_contact_like(chosen):
            chosen = "contacts"
        topic_label_map[topic_id] = chosen
    used_in_cat = {}
    for idx, page in enumerate(pages):
        tid = int(topics[idx])
        cat = topic_label_map.get(tid, "general")
        cat_dir = domain_dir / cat
        cat_dir.mkdir(parents=True, exist_ok=True)
        base_slug = slug_from_title(page["title"], page["url"])
        used = used_in_cat.setdefault(cat, set())
        name = base_slug
        n = 1
        while name in used:
            n += 1
            name = f"{base_slug}_{n}"
        used.add(name)
        path = cat_dir / f"{name}.md"
        content = f"Source: {page['url']}\n\n# {page['title']}\n\n{page['text']}"
        path.write_text(content, encoding="utf-8")
    n_cats = len(used_in_cat)
    print(f"Saved {len(pages)} pages under {domain_dir}: {n_cats} content-based categories")

domain = get_domain(BASE_URL)
domain_dir = DATA_SOURCE / domain
domain_dir.mkdir(parents=True, exist_ok=True)

md_files = [p for p in domain_dir.glob("**/*.md") if p.name != "_summary.md"]
if md_files:
    print(f"Re-categorizing {len(md_files)} existing pages at {domain_dir} (content-based)...")
    pages = load_pages_from_md(domain_dir)
    for subdir in domain_dir.iterdir():
        if subdir.is_dir():
            shutil.rmtree(subdir)
    if not pages:
        pages = [{"url": BASE_URL, "title": "Demo", "text": "No content to re-categorize."}]
        (domain_dir / "general").mkdir(exist_ok=True)
        (domain_dir / "general" / "demo.md").write_text(
            f"Source: {BASE_URL}\n\n# Demo\n\n{pages[0]['text']}", encoding="utf-8"
        )
        print("No pages loaded; created general/demo.md")
    else:
        categorize_and_save(pages, domain_dir)
else:
    max_links = 200
    crawl_workers = 8
    print(f"Crawling {BASE_URL} (max {max_links} links, same-domain only) with {crawl_workers} workers...")
    with tqdm(total=max_links, desc="Crawled", unit="page") as pbar:
        def on_progress(n, max_l, q):
            pbar.n = min(n, max_l)
            pbar.set_postfix(queue=q)
            pbar.refresh()
        pages, stats = crawl_site(BASE_URL, output_dir=domain_dir, max_links=max_links, progress_callback=on_progress, workers=crawl_workers)
    print(f"Pages visited: {stats['total_pages_visited']}, Links discovered: {stats['total_links_discovered']}")
    if not pages:
        fallback_dir = domain_dir / "general"
        fallback_dir.mkdir(exist_ok=True)
        (fallback_dir / "demo.md").write_text(
            "Source: https://example.com\n\n# Demo\n\nNo pages could be crawled. Using demo content for RAG.",
            encoding="utf-8",
        )
        print("No pages crawled; created demo.md for RAG.")
    else:
        categorize_and_save(pages, domain_dir)

if (domain_dir / "_summary.md").exists():
    print((domain_dir / "_summary.md").read_text(encoding="utf-8"))

## Cell 4 — Document Processing & Embedding

### List source files and total content size

In [ ]:
import glob

base_url_path = get_domain(BASE_URL)
site_content_path = f"data/source/{base_url_path}/**/*.md"
files = glob.glob(site_content_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

all_site_content = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        all_site_content += f.read()
        all_site_content += "\n\n"

print(f"Total characters in site content pages: {len(all_site_content):,}")

### Token count for loaded content

In [ ]:
# How many tokens in all the documents?
import tiktoken

encoding = tiktoken.encoding_for_model(DEFAULT_MODEL)
tokens = encoding.encode(all_site_content)
token_count = len(tokens)
print(f"Total tokens for {DEFAULT_MODEL}: {token_count:,}")

### Load documents from data/source

In [ ]:
from langchain_community.docstore.document import Document
from langchain_community.document_loaders import DirectoryLoader, TextLoader


folders = glob.glob(f"data/source/{base_url_path}/*")


documents = []
for folder in folders:
    if folder.endswith(".md"):
        print("Not a directory")
        continue
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        if doc.metadata["source"].endswith("_summary.md"):
                continue
            
        url = ""
        if doc.page_content.startswith("Source: "):
            first_line, _, body = doc.page_content.partition("\n\n")
            

            url = first_line.replace("Source: ", "").strip()
            doc.page_content = body
        doc.metadata["url"] = url
        documents.append(doc)

print(f"Loaded {len(documents)} documents from {len(folders)} folders")

print(documents[0].metadata)


### Chunk documents (LLM or RecursiveCharacterTextSplitter)

In [ ]:
if USE_LLM_CHUNKING:
    print("Using LLM chunking")

    chunking_client, chunk_model = get_openai_client("chunking")
    import json
    from concurrent.futures import ThreadPoolExecutor, as_completed

    def chunk_doc(args):
        i, doc = args
        meta = doc.metadata
        how_many = max(1, len(doc.page_content) // CHUNK_SIZE)
        prompt = f"""
        You take a document and you split the document into overlapping chunks for a KnowledgeBase.

        The document is from the website of a school or institution.
        The document has been retrieved from: {meta["source"]}

        A chatbot will use these chunks to answer questions about the institution.
        You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
        This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
        There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

        For each chunk, you should provide a headline, a summary, and the original text of the chunk.
        Together your chunks should represent the entire document with overlap.

        Here is the document:

        {doc.page_content[:12000]}

        Respond with the chunks in JSON only: {{"chunks": [{{"headline": "...", "summary": "...", "original_text": "..."}}, ...]}}.
        """
        resp = chunking_client.chat.completions.create(
            model=chunk_model,
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"}
        )
        data = json.loads(resp.choices[0].message.content)
        return i, doc, data

    chunks = []
    MAX_WORKERS = 8  # tune this; won't exceed API rate limits for small batches
    results = {}

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(chunk_doc, (i, doc)): i for i, doc in enumerate(documents)}
        for future in as_completed(futures):
            i, doc, data = future.result()
            results[i] = (doc, data)
            done = len(results)
            if done % 5 == 0 or done == 1 or done == len(documents):
                print(f"LLM chunking doc {done}/{len(documents)}...")

    # Rebuild in original order
    for i in sorted(results):
        doc, data = results[i]
        meta = doc.metadata
        for c in data.get("chunks", []):
            content = f"{c.get('headline', '')}\n\n{c.get('summary', '')}\n\n{c.get('original_text', '')}"
            chunks.append(Document(page_content=content, metadata=dict(meta)))

else:
    print("Using default chunking with character splitter")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        length_function=len,
    )
    chunks = text_splitter.split_documents(documents)




### Preview chunks

In [ ]:
print(f"✂️ Split into {len(chunks)} chunks")

for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i} ---")
    print(f"Source: {chunk.metadata.get('source')}, URL: {chunk.metadata.get('url', '')[:50]}...")
    print(f"Content: {chunk.page_content[:200]}...")

## Cell 5 — Chroma Vector Store

In [ ]:
from chromadb import PersistentClient

DB_PATH.mkdir(parents=True, exist_ok=True)
client = PersistentClient(path=str(DB_PATH))

def load_store():
    existing_store = Chroma(client=client, collection_name=COLLECTION_NAME, embedding_function=embeddings)
    return existing_store

def populate_store():
    try:
        client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass
    print("🆕 Building vector store from chunks...")
    new_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        client=client,
        collection_name=COLLECTION_NAME,
    )

    return new_store

try:
    if chunks:
       vectorstore = populate_store()
    else:
       vectorstore =  load_store()
except NameError:
    vectorstore = load_store()

print(f"✅ Vector store ready at {DB_PATH} with {vectorstore._collection.count()} embeddings")

## Cell 6 — Retrieval & Answer Generation

### Rerank, retrieve_and_answer, and streaming

Retrieve top-k chunks, optionally re-rank with LLM, then generate answer with context. Streaming used by the Chat UI.

In [ ]:
chat_client, chat_model = get_openai_client("chat")

RAG_SYSTEM_PROMPT = """You are a helpful assistant that uses the knowledge base of a website for a school or institution
to interact with site visitors and answer their questions, responding in a concise professional way and sharing reference
website resources. If you don't know the answer to the question, say so.

Context from website knowledge base:
{context}
"""

def rerank(query: str, docs: list, top_k: int) -> list:
    from pydantic import BaseModel, Field
    class RankOrder(BaseModel):
        order: list[int] = Field(description="Ranked chunk IDs from 1 to N, most relevant first")
    user_prompt = f"Question:\n{query}\n\nChunks (rank by relevance to the question, most relevant first):\n"
    for i, doc in enumerate(docs):
        user_prompt += f"# CHUNK ID {i + 1}:\n{doc.page_content[:1500]}\n\n"
    user_prompt += "Reply with JSON only: {\"order\": [list of chunk IDs in relevance order]}."

    rerank_client, rerank_model = get_openai_client("reranking")
    response = rerank_client.chat.completions.create(
        model=rerank_model,
        messages=[{"role": "user", "content": user_prompt}],
        response_format={"type": "json_object"}
    )
    import json
    data = json.loads(response.choices[0].message.content)
    order = data.get("order", list(range(1, len(docs) + 1)))
    return [docs[i - 1] for i in order if 1 <= i <= len(docs)][:top_k]


def retrieve_and_answer(query: str, top_k: int | None = None) -> dict:
    top_k = top_k or RERANK_TOP_K
    docs = vectorstore.similarity_search(query, k=RETRIEVAL_K)
    if len(docs) > top_k:
        docs = rerank(query, docs, top_k)
    else:
        docs = docs[:top_k]
    context = "\n\n".join([doc.page_content for doc in docs])
    sources = [doc.metadata.get("url") or doc.metadata.get("source", "") for doc in docs]
    
    messages = [
        {"role": "system", "content": RAG_SYSTEM_PROMPT.format(context=context)},
        {"role": "user", "content": query}
    ]
    
    response = chat_client.chat.completions.create(
        model=chat_model,
        messages=messages
    )
    answer = response.choices[0].message.content
    unique_sources = list(dict.fromkeys(sources))
    sources_text = "\n".join([f"- {s}" for s in unique_sources if s])
    return {
        "answer": answer,
        "sources": sources_text,
        "num_chunks_retrieved": len(docs),
        "docs": docs,
    }


def _retrieve_docs(query: str, top_k: int | None = None):
    top_k = top_k or RERANK_TOP_K
    docs = vectorstore.similarity_search(query, k=RETRIEVAL_K)
    if len(docs) > top_k:
        docs = rerank(query, docs, top_k)
    else:
        docs = docs[:top_k]
    return docs


def stream_rag_response(query: str):
    docs = _retrieve_docs(query)
    context = "\n\n".join([doc.page_content for doc in docs])
    messages = [
        {"role": "system", "content": RAG_SYSTEM_PROMPT.format(context=context)},
        {"role": "user", "content": query},
    ]
    chat_client, chat_model = get_openai_client("chat")
    stream = chat_client.chat.completions.create(
        model=chat_model,
        messages=messages,
        stream=True,
    )
    acc = ""
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            acc += chunk.choices[0].delta.content
            yield acc, docs
    yield acc, docs




# Test the pipeline
# test_query = "What are the important things to know about this institution or school"
# print(f"❓ Query: {test_query}\n")

# result = retrieve_and_answer(test_query)
# print(f"💬 Answer:\n{result['answer']}")
# print(f"\n📚 Sources:\n{result['sources']}")

print("Retrieval layer is ready!")

### Generate evaluation tests from vectorstore (optional)

Run this cell to build test questions from the current knowledge base; then run the evaluation cell below. Uses the vectorstore to sample chunks and an LLM to produce question, keywords, reference answer, and category. Optionally saves to `evaluation/tests.jsonl` so Gradio and file-based flows see the same set.

In [ ]:
from evaluation.test import load_tests, save_tests
from evaluation.generate_tests import generate_tests_from_vectorstore
from evaluation.eval import run_eval_summary

eval_client, eval_model = get_openai_client("chat")
generated_tests = generate_tests_from_vectorstore(
    vectorstore, eval_client, eval_model, max_tests=10, seed=42
)
if generated_tests:
    save_tests(generated_tests)
    print(f"Generated and saved {len(generated_tests)} tests to evaluation/tests.jsonl")

## Cell 6b — Evaluation

Run retrieval and answer evals over `evaluation/tests.jsonl`. Requires vectorstore and `retrieve_and_answer` from above.

In [ ]:
from evaluation.test import load_tests
from evaluation.eval import evaluate_retrieval, evaluate_answer, run_eval_summary

def eval_fetch(question: str, k: int = 10):
    return vectorstore.similarity_search(question, k=k)

def eval_get_answer(question: str) -> str:
    return retrieve_and_answer(question)["answer"]
    
eval_tests = load_tests()
eval_client, eval_model = get_openai_client("chat")
summary = run_eval_summary(
    eval_fetch, eval_get_answer, k_retrieval=10,
    judge_client=eval_client, judge_model=eval_model,
    tests=eval_tests, max_workers=4
)
print("Evaluation summary (RAG pipeline)")
print("Retrieval — avg MRR:", round(summary["retrieval"]["avg_mrr"], 4), "| avg keyword coverage %:", round(summary["retrieval"]["avg_keyword_coverage_pct"], 1))
print("Answer — avg accuracy:", round(summary["answer"]["avg_accuracy"], 2), "| completeness:", round(summary["answer"]["avg_completeness"], 2), "| relevance:", round(summary["answer"]["avg_relevance"], 2))
print("n_tests:", summary["n_tests"])

### Generate welcome message from knowledge base

In [ ]:
def generate_welcome_message(vectorstore, client, model: str) -> str:
    """Generate a welcome message using the knowledge base URLs to infer topics."""
    result = vectorstore._collection.get(include=["metadatas"])
    metadatas = result.get("metadatas") or []
    urls = list(dict.fromkeys(
        (m.get("url") or m.get("source") or "").strip()
        for m in metadatas
        if (m.get("url") or m.get("source") or "").strip()
    ))[:30]
    if not urls:
        return "👋 Hi! I'm **Joe**. Ask me anything about this institution — I'll cite my sources!"

    links_text = "\n".join(urls)

    prompt = f"""You are writing a welcome message for a chat assistant called "Joe".
Joe is built on a knowledge base crawled from a website. Here are the page URLs it covers:

{links_text}

From these URLs, identify:
- The institution or organisation name (from the domain or path patterns)
- The main topic areas covered (infer from URL path segments like /fees/, /admissions/, /about/, etc.)

Then write a welcome message in EXACTLY this format:

---
👋 Hi! I'm **Joe**, your assistant for **[Institution Name]**.

I can help you with:
- 🎓 **[Topic]** — [one-line description]
- 💰 **[Topic]** — [one-line description]
- 📞 **[Topic]** — [one-line description]
- 🏫 **[Topic]** — [one-line description]
- 🖥️ **[Topic]** — [one-line description]

Feel free to ask me anything — I'll always cite my sources!
---

Rules:
- Each topic on its OWN LINE starting with a dash (-)
- 4-6 topics only, based on what the URLs actually cover
- Return ONLY the message, no extra commentary
"""
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    return resp.choices[0].message.content.strip()



print("⏳ Generating welcome message from knowledge base…")
_chat_client, _chat_model = get_openai_client("chat")
WELCOME_MESSAGE = generate_welcome_message(vectorstore, _chat_client, _chat_model)
print("✅ Welcome message ready:\n")
print(WELCOME_MESSAGE)


## Cell 7 — Gradio UI

Chat tab with streaming, Relevant Articles tab for source docs and links, and Evaluation tab to run retrieval/answer metrics.

In [ ]:
import gradio as gr
import pandas as pd
from evaluation.eval import evaluate_retrieval, evaluate_answer
from evaluation.test import load_tests


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def format_relevant_articles(docs: list) -> str:
    if not docs:
        return "_Send a message to see relevant articles here._"
    
    seen_urls = set()
    lines = []
    for doc in docs:
        url    = (doc.metadata.get("url")    or "").strip()
        source = (doc.metadata.get("source") or "").strip()

        # Skip duplicate URLs
        key = url or source
        if key in seen_urls:
            continue
        seen_urls.add(key)

        # Parse LLM chunk structure: headline \n\n summary \n\n original_text
        parts = doc.page_content.split("\n\n", 2)
        headline = parts[0].strip() if len(parts) > 0 else ""
        summary  = parts[1].strip() if len(parts) > 1 else (doc.page_content[:180].replace("\n", " ") + "…")

        title = headline or source or url or "Article"
        link  = f"[{title}]({url})" if url else f"**{title}**"
        lines.append(f"### {link}\n{summary}\n")

    return "\n---\n".join(lines) if lines else "_No sources found._"



# ---------------------------------------------------------------------------
# Chat handlers  — two-step: (1) add user msg + clear box, (2) stream reply
# ---------------------------------------------------------------------------

def add_user_msg(message: str, history: list) -> tuple[str, list]:
    """Append user turn to history and immediately clear the textbox."""
    if not message.strip():
        return message, history
    return "", history + [{"role": "user", "content": message}]


def stream_bot_reply(history: list, docs_state: list):
    if not history or history[-1]["role"] != "user":
        yield history, docs_state, format_relevant_articles(docs_state)
        return
    user_msg = history[-1]["content"]
    history  = history + [{"role": "assistant", "content": ""}]
    docs = docs_state
    for partial, docs in stream_rag_response(user_msg):
        history[-1]["content"] = partial
        yield history, docs, format_relevant_articles(docs)
    yield history, docs, format_relevant_articles(docs)



# ---------------------------------------------------------------------------
# Evaluation handler
# ---------------------------------------------------------------------------

def run_evaluation(progress=gr.Progress()):
    tests = load_tests()
    n = len(tests)
    if n == 0:
        return "<p>No tests found in evaluation/tests.jsonl</p>", None, None

    ret_mrrs, ret_cov = [], []
    ans_acc, ans_comp, ans_rel = [], [], []
    by_cat_mrr: dict = {}
    by_cat_acc: dict = {}

    for i, test in enumerate(tests):
        progress((i + 1) / n, desc=f"Evaluating {i + 1}/{n}…")
        ret = evaluate_retrieval(test, eval_fetch, k=10)
        ans_eval, _ = evaluate_answer(test, lambda q: eval_get_answer(q))
        ret_mrrs.append(ret.mrr);  ret_cov.append(ret.keyword_coverage)
        ans_acc.append(ans_eval.accuracy);  ans_comp.append(ans_eval.completeness);  ans_rel.append(ans_eval.relevance)
        by_cat_mrr.setdefault(test.category, []).append(ret.mrr)
        by_cat_acc.setdefault(test.category, []).append(ans_eval.accuracy)

    def color(v, green, amber):
        return "green" if v >= green else ("orange" if v >= amber else "red")

    avg_mrr  = sum(ret_mrrs) / n
    avg_cov  = sum(ret_cov)  / n
    avg_acc  = sum(ans_acc)  / n
    avg_comp = sum(ans_comp) / n
    avg_rel  = sum(ans_rel)  / n

    cards = [
        ("MRR",             avg_mrr,  0.90, 0.75, f"{avg_mrr:.4f}"),
        ("Keyword Cov.",    avg_cov,  90,   75,   f"{avg_cov:.1f}%"),
        ("Accuracy /5",     avg_acc,  4.5,  4.0,  f"{avg_acc:.2f}"),
        ("Completeness /5", avg_comp, 4.5,  4.0,  f"{avg_comp:.2f}"),
        ("Relevance /5",    avg_rel,  4.5,  4.0,  f"{avg_rel:.2f}"),
    ]
    card_html = "".join(
        f'<div style="padding:16px;background:#1a1a2e;border-radius:12px;'
        f'border-left:6px solid {color(v,g,a)};flex:1;min-width:130px">'
        f'<div style="font-size:11px;text-transform:uppercase;letter-spacing:1px;color:#9090b0;margin-bottom:6px">{lbl}</div>'
        f'<div style="font-size:28px;font-weight:800;color:#ffffff;line-height:1">{disp}</div>'
        f'</div>'
        for lbl, v, g, a, disp in cards
    )
    html = (
        f'<div style="display:flex;flex-wrap:wrap;gap:12px;padding:10px">{card_html}</div>'
        f'<p style="padding:4px 10px;color:#5cb85c;font-weight:600">✅ {n} tests completed</p>'
    )

    df_mrr = pd.DataFrame([{"Category": c, "Avg MRR": round(sum(s)/len(s), 4)} for c, s in by_cat_mrr.items()])
    df_acc = pd.DataFrame([{"Category": c, "Avg Accuracy": round(sum(s)/len(s), 4)} for c, s in by_cat_acc.items()])
    return html, df_mrr, df_acc


# ---------------------------------------------------------------------------
# Layout
# ---------------------------------------------------------------------------

with gr.Blocks(title="Institution Website RAG Chatbot", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Institution Website RAG Chatbot")
    gr.Markdown("Chat with the institution's website knowledge base. Ask about admissions, fees, programs, or anything on the site — answers cite source pages.")
    docs_state = gr.State(value=[])

    # ── Main: full-width chat ───────────────────────────────────────────────
    chatbot = gr.Chatbot(
        label="Chat",
        height=500,
        type="messages",          # modern bubbles format
        show_copy_button=True,
        bubble_full_width=False,
        value=[{"role": "assistant", "content": WELCOME_MESSAGE}]
    )
    with gr.Row():
        msg_in = gr.Textbox(
            placeholder="Ask about admissions, fees, programs, or anything on the institution's website…",
            show_label=False,
            scale=8,
            lines=1,
            max_lines=5,
        )
        send_btn  = gr.Button("Send ➤", variant="primary", scale=1, min_width=90)
        clear_btn = gr.ClearButton([chatbot, docs_state], value="🗑 Clear", scale=1, min_width=80)

    # ── Below: Sources + Evaluation tabs ───────────────────────────────────
    with gr.Tabs():
        with gr.TabItem("📰 Relevant Articles"):
            sources_out = gr.Markdown("_Send a message to see relevant articles here._")

        with gr.TabItem("📊 Evaluation"):
            eval_btn = gr.Button("▶ Run Evaluation", variant="primary", size="lg")
            with gr.Row():
                eval_metrics   = gr.HTML("<div style='padding:20px;color:#999'>Click Run Evaluation to start.</div>")
                with gr.Column():
                    eval_chart_mrr = gr.BarPlot(x="Category", y="Avg MRR",      title="MRR by Category",      y_lim=[0, 1], height=260)
                    eval_chart_acc = gr.BarPlot(x="Category", y="Avg Accuracy",  title="Accuracy by Category", y_lim=[1, 5], height=260)
            eval_btn.click(run_evaluation, outputs=[eval_metrics, eval_chart_mrr, eval_chart_acc])

    # ── Event wiring ────────────────────────────────────────────────────────
    # Both submit (Enter key) and Send button follow the same two-step chain:
    #   Step 1 — add_user_msg  → clears textbox, appends user bubble instantly
    #   Step 2 — stream_bot_reply → streams reply + live-updates Sources tab
    for trigger in (msg_in.submit, send_btn.click):
        trigger(
            add_user_msg,
            inputs=[msg_in, chatbot],
            outputs=[msg_in, chatbot],
            queue=False,
        ).then(
            stream_bot_reply,
            inputs=[chatbot, docs_state],
            outputs=[chatbot, docs_state, sources_out],
        )

demo.launch(inbrowser=True)


---

## Concepts Demonstrated

| Cell | Pipeline / API | What it shows |
|------|---------------|---------------|
| 3 | Crawl + categorize | Same-domain crawl of institution site, BERTopic zero-shot categories, data/source |
| 4 | Document splitting | Load institution site content from data/source; RecursiveCharacterTextSplitter or LLM chunking (USE_LLM_CHUNKING) |
| 5 | Chroma DB | Vector storage in data/db |
| 6 | RAG pipeline | Retrieve RETRIEVAL_K, LLM re-rank to RERANK_TOP_K, then generate; citations for learners |
| 6b | Evaluation | Retrieval (MRR, keyword coverage) + answer (LLM-as-judge) over evaluation/tests.jsonl |
| 7 | Gradio UI | Chat for learners (streaming), Relevant Articles tab (source docs + links), Evaluation tab (Run + charts) |

---

## Extensions Possible

1. **Hybrid search**: Combine semantic + keyword search
2. **Query rewriting**: Expand/rewrite query before retrieval
3. **Multi-modal**: Support PDFs, images
4. **Custom embeddings**: Use local embedding models
